# Taiwanese Bankruptcy Prediction using Logistic Regression

---

## Project Overview
This project analyzes the Taiwanese Bankruptcy Prediction dataset from the UCI Machine Learning Repository and builds a Logistic Regression classification model to predict company bankruptcy.

**Dataset Information:**
- **Source:** Taiwan Economic Journal (1999-2009)
- **Instances:** 6,819 companies
- **Features:** 95 financial indicators
- **Target:** Binary classification (0 = Non-bankrupt, 1 = Bankrupt)

**Objectives:**
1. Explore and analyze the dataset
2. Handle class imbalance
3. Build and evaluate Logistic Regression model
4. Interpret feature importance for business insights

---
## 1. Import Libraries

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


---
## 2. Load Dataset

In [4]:
taiwanese_bankruptcy = fetch_ucirepo(id=572)
X = taiwanese_bankruptcy.data.features
y = taiwanese_bankruptcy.data.targets
df = pd.concat([y, X], axis=1)

---
## 3. Initial Data Exploration

### 3.1 Display First Few Rows

In [5]:
df.sample(5)

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
603,0,0.508409,0.579481,0.565287,0.602481,0.602466,0.999003,0.797457,0.809379,0.303571,...,0.816231,0.002819,0.623586,0.602477,0.841675,0.279680,0.026988,0.565949,1,0.029597
2091,0,0.473017,0.525894,0.520745,0.600095,0.600095,0.998954,0.797346,0.809275,0.303479,...,0.790793,0.000662,0.623529,0.600094,0.839696,0.277686,0.026184,0.558357,1,0.042444
1169,0,0.485107,0.543393,0.531881,0.607201,0.607201,0.999023,0.797429,0.809342,0.303481,...,0.799942,0.004357,0.623638,0.607201,0.840351,0.279274,0.027661,0.567434,1,0.031310
3505,0,0.559109,0.607992,0.623802,0.609125,0.609125,0.999143,0.797587,0.809509,0.303504,...,0.836350,0.003633,0.623421,0.609127,0.842861,0.278129,0.026832,0.565347,1,0.038305
2769,0,0.542290,0.572285,0.578939,0.624461,0.624461,0.999137,0.797573,0.809433,0.303494,...,0.818764,0.003898,0.623693,0.624456,0.841651,0.278238,0.026798,0.565189,1,0.037451


### 3.2 Dataset Information

In [7]:
df.shape

(6819, 96)

### 3.3 Statistical Summary

In [8]:
df.describe()

,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
count,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,...,6819.000000,6.819000e+03,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.000000,6819.0,6819.000000
mean,0.032263,0.505180,0.558625,0.553589,0.607948,0.607929,0.998755,0.797190,0.809084,0.303623,...,0.807760,1.862942e+07,0.623915,0.607946,0.840402,0.280365,0.027541,0.565358,1.0,0.047578
std,0.176710,0.060686,0.065620,0.061595,0.016934,0.016916,0.013010,0.012869,0.013601,0.011163,...,0.040332,3.764501e+08,0.012290,0.016934,0.014523,0.014463,0.015668,0.013214,0.0,0.050014
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000
25%,0.000000,0.476527,0.535543,0.527277,0.600445,0.600434,0.998969,0.797386,0.809312,0.303466,...,0.796750,9.036205e-04,0.623636,0.600443,0.840115,0.276944,0.026791,0.565158,1.0,0.024477
50%,0.000000,0.502706,0.559802,0.552278,0.605997,0.605976,0.999022,0.797464,0.809375,0.303525,...,0.810619,2.085213e-03,0.623879,0.605998,0.841179,0.278778,0.026808,0.565252,1.0,0.033798
75%,0.000000,0.535563,0.589157,0.584105,0.613914,0.613842,0.999095,0.797579,0.809469,0.303585,...,0.826455,5.269777e-03,0.624168,0.613913,0.842357,0.281449,0.026913,0.565725,1.0,0.052838
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,9.820000e+09,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000


### 3.4 Check for Missing Values

In [9]:
df.isnull().sum()

Bankrupt?                                                   0
 ROA(C) before interest and depreciation before interest    0
 ROA(A) before interest and % after tax                     0
 ROA(B) before interest and depreciation after tax          0
 Operating Gross Margin                                     0
                                                           ..
 Liability to Equity                                        0
 Degree of Financial Leverage (DFL)                         0
 Interest Coverage Ratio (Interest expense to EBIT)         0
 Net Income Flag                                            0
 Equity to Liability                                        0
Length: 96, dtype: int64

---
## 4. Target Variable Analysis

### 4.1 Class Distribution

### 4.2 Visualize Class Distribution

---
## 5. Exploratory Data Analysis (EDA)

### 5.1 Feature Distributions

### 5.2 Correlation Analysis

### 5.3 Feature Distribution by Bankruptcy Status

---
## 6. Data Preprocessing

### 6.1 Train-Test Split

### 6.2 Feature Scaling

### 6.3 Handle Class Imbalance (SMOTE)

---
## 7. Model Training - Logistic Regression

### 7.1 Train Logistic Regression Model

### 7.2 Cross-Validation

---
## 8. Model Predictions

---
## 9. Model Evaluation

### 9.1 Performance Metrics

### 9.2 Classification Report

---
## 10. Confusion Matrix Visualization

---
## 11. ROC Curve and Precision-Recall Curve

---
## 12. Feature Importance Analysis

### 12.1 Extract Feature Coefficients

### 12.2 Visualize Top Features

---
## 13. Model Summary and Insights

---
## 14. Conclusions

### Key Findings:
- [To be filled after analysis]

### Model Performance:
- [To be filled after evaluation]

### Business Recommendations:
- [To be filled after interpretation]

### Future Work:
- Compare with other algorithms (Random Forest, XGBoost, SVM)
- Hyperparameter tuning
- Feature engineering
- Ensemble methods